## 🧠 Conversational RAG (LangChain + Ollama)

This notebook explores building a **local Retrieval-Augmented Generation (RAG)** system using:

- LangChain  
- Ollama (Llama 3.2)  
- HuggingFace embeddings  
- Chroma vector database  
- Gradio UI  

### What this notebook does

- Loads a custom knowledge base (markdown files)
- Splits documents into chunks
- Generates embeddings locally (CPU)
- Stores them in Chroma (persistent vector DB)
- Retrieves relevant chunks for a query
- Uses a local LLM to generate grounded answers
- Supports **multi-turn conversations (chat history)**

---

### Key learning

This notebook was built after an earlier **manual RAG pipeline**, which had issues with:

- retrieval quality  
- follow-up questions  
- pipeline complexity  

This version focuses on:

- cleaner implementation using LangChain  
- handling conversational queries  
- introducing **query rewriting for follow-ups**

---

### Limitations

- Retrieval is semantic, not structured
- Questions like:
  - "Who is the youngest employee?"
  - "Who is the worst performer?"
  
  may fail because they require **aggregation or reasoning**, not just retrieval

---

### Example queries

- Who is the CEO of Insurellm?
- Who won the IIOTY award?
- Tell me more about her performance
- When did she join the company?

---

### Notes

- Fully local setup (no external APIs)
- Vector DB is stored on disk
- Designed for experimentation and learning, not production

---

### Next steps

- Improve retrieval with reranking  
- Add history-aware retrieval  
- Explore agent-based reasoning  

In [1]:
from pathlib import Path

PROJECT_ROOT = Path("/mnt/data/llm_project/llm-text-learning-lab")
KNOWLEDGE_BASE = PROJECT_ROOT / "data"/"knowledge-base"
VECTOR_DB = PROJECT_ROOT / "vector_db"

COLLECTION_NAME = "insurellm_docs"

OLLAMA_MODEL = "llama3.2:latest"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

CHUNK_SIZE = 700
CHUNK_OVERLAP = 150
RETRIEVAL_K = 10

EMBED_BATCH_SIZE = 8      #
ADD_BATCH_SIZE = 64  

In [2]:
from pathlib import Path
import shutil

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
def load_markdown_docs(knowledge_base_path: Path) -> list[Document]:
    docs = []

    for folder in knowledge_base_path.iterdir():
        if not folder.is_dir():
            continue

        doc_type = folder.name

        for file in folder.rglob("*.md"):
            text = file.read_text(encoding="utf-8").strip()

            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": file.as_posix(),
                        "doc_type": doc_type,
                        "source_name": file.stem.replace("_", " ").replace("-", " "),
                    },
                )
            )

    return docs


docs = load_markdown_docs(KNOWLEDGE_BASE)
print("Loaded docs:", len(docs))

Loaded docs: 84


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

split_docs = splitter.split_documents(docs)
print("Split docs:", len(split_docs))

Split docs: 688


In [5]:
if VECTOR_DB.exists():
    shutil.rmtree(VECTOR_DB)

VECTOR_DB.mkdir(parents=True, exist_ok=True)
print("Fresh vector DB folder ready:", VECTOR_DB)

Fresh vector DB folder ready: /mnt/data/llm_project/llm-text-learning-lab/vector_db


In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": EMBED_BATCH_SIZE,
    },
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(VECTOR_DB),
)

for start in range(0, len(split_docs), ADD_BATCH_SIZE):
    end = start + ADD_BATCH_SIZE
    batch = split_docs[start:end]
    vectorstore.add_documents(batch)
    print(f"Added {min(end, len(split_docs))}/{len(split_docs)} chunks")

print("Vector DB stored at:", VECTOR_DB)

Added 64/688 chunks
Added 128/688 chunks
Added 192/688 chunks


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
)

In [ ]:
def rewrite_query_with_history(question, history):
    if not history:
        return question

    history_text = "\n".join(
        f"{m.content}" for m in history if hasattr(m, "content")
    )

    prompt = f"""
You are helping retrieve documents from a knowledge base.

Conversation history:
{history_text}

Current question:
{question}

Rewrite the question into a standalone question with full names instead of pronouns.

Example:
"Tell me more about her" → "Tell me more about Maxine Thompson"

Only return the rewritten question.
"""

    response = llm.invoke(prompt)
    return response.content.strip()

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable assistant.

Rules:
- Use ONLY the provided context
- You MAY combine information across multiple documents
- You MAY perform comparisons (e.g., youngest, best, worst)
- If the answer cannot be derived, say:
I could not find that in the knowledge base.
""".strip()


qa_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])


def format_context(docs):
    return "\n\n".join(
        f"Source: {d.metadata.get('source_name', d.metadata.get('source', 'unknown'))}\n{d.page_content}"
        for d in docs
    )


def answer_question(question, history=None):
    history = history or []

    rewritten_question = rewrite_query_with_history(question, history)
    retrieved_docs = retriever.invoke(rewritten_question)
    context = format_context(retrieved_docs)

    messages = qa_prompt.invoke({
        "history": history,
        "context": context,
        "question": question,
    })

    response = llm.invoke(messages)
    return response.content, retrieved_docs

In [ ]:
answer, retrieved_docs = answer_question("Who is the CEO of Insurellm?")
print(answer)

In [ ]:
answer, retrieved_docs = answer_question("Who went to the manchester university?")
print(answer)

In [ ]:
import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

def gradio_history_to_langchain(history):
    lc_history = []

    if not history:
        return lc_history

    for item in history:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")

            if not content:
                continue

            if role == "user":
                lc_history.append(HumanMessage(content=content))
            elif role == "assistant":
                lc_history.append(AIMessage(content=content))

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_msg, assistant_msg = item

            if user_msg:
                lc_history.append(HumanMessage(content=user_msg))
            if assistant_msg:
                lc_history.append(AIMessage(content=assistant_msg))

    return lc_history


def chat_fn(message, history):
    lc_history = gradio_history_to_langchain(history)

    answer, _ = answer_question(message, history=lc_history)

    return answer


demo = gr.ChatInterface(
    fn=chat_fn,
    title="Insurellm RAG Chat",
    description="Ask questions about the Insurellm knowledge base.",
)

demo.launch()